In [1]:
print(1)

1


In [2]:
import anndata as ad
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [3]:
import matplotlib.pyplot as plt

In [4]:
from sklearn.decomposition import PCA

In [5]:
import dilimap as dmap

In [6]:
import pickle

# Download_data

In [7]:
l1000_phase1_path = '/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/l1000_phase1/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/'
l1000_phase1_files = os.listdir(l1000_phase1_path)

l1000_phase2_path = '/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/l1000_phase2/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/'
l1000_phase2_files = os.listdir(l1000_phase2_path)

In [8]:
l1000_phase1 = []
for file in tqdm(l1000_phase1_files):
    l1000_phase1.append(ad.read_h5ad(l1000_phase1_path + file))

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_v

In [9]:
l1000_phase2 = []
for file in tqdm(l1000_phase2_files):
    l1000_phase2.append(ad.read_h5ad(l1000_phase2_path + file))

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_v

In [10]:
dili_train_path = '../../../op3_v2/data/dilimap_train_val/raw/adata_training_counts.h5ad'
dili_val_path = '../../../op3_v2/data/dilimap_train_val/raw/adata_validation_counts.h5ad'

dili_train = ad.read_h5ad(dili_train_path)
dili_val = ad.read_h5ad(dili_val_path)

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [11]:
with open("../../dili.pkl", "rb") as file:
    df_dili = pickle.load(file)

In [12]:
df_compounds_train = dili_train.obs[['COMPOUND', 'CONCENTRATION_UM', 'TIMEPOINT_HOURS', 'SPLIT']]
df_compounds_val = dili_val.obs[['COMPOUND', 'CONCENTRATION_UM', 'TIMEPOINT_HOURS', 'SPLIT']]

In [13]:
df_compounds = pd.concat([df_compounds_train, df_compounds_val])

In [14]:
df_compounds = df_compounds.merge(df_dili[['pert_id', 'pubchem_cid']], how='left', left_on='COMPOUND', right_on='pert_id')

In [15]:
df_compounds.loc[df_compounds['COMPOUND'] == 'DMSO', 'pubchem_cid'] = 679

In [16]:
df_compounds['pubchem_cid'] = df_compounds['pubchem_cid'].fillna(-666).astype(int).astype(str).astype('category').replace({'-666': None, -666: None})

/home/icb/olga.novitskaia/tmp/ipykernel_477545/269742383.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_compounds['pubchem_cid'] = df_compounds['pubchem_cid'].fillna(-666).astype(int).astype(str).astype('category').replace({'-666': None, -666: None})


# Overlapping compounds

In [17]:
df_compounds_l1000 = pd.read_csv('../../df_compounds.csv')

In [18]:
DILI_TIME = 24.0  
overlapping_compounds1 = {}
for j, l1000_phase1_j in enumerate(l1000_phase1):
    l1000_24h = l1000_phase1_j.obs[l1000_phase1_j.obs['pert_time_h'] == DILI_TIME]
    overlapping_compounds1[j] = len(set(df_compounds['pubchem_cid']).intersection(set(l1000_24h['pubchem_cid'])))

In [19]:
overlapping_compounds2 = {}
for j, l1000_phase2_j in enumerate(l1000_phase2):
    l1000_24h = l1000_phase2_j.obs[l1000_phase2_j.obs['pert_time_h'] == DILI_TIME]
    overlapping_compounds2[j] = len(set(df_compounds['pubchem_cid']).intersection(set(l1000_24h['pubchem_cid'])))

In [20]:
sorted(overlapping_compounds1.items(), key=lambda item: item[1], reverse=True)[:10]

[(47, 165),
 (14, 164),
 (25, 150),
 (54, 133),
 (5, 107),
 (37, 98),
 (62, 94),
 (0, 90),
 (43, 90),
 (7, 77)]

In [21]:
sorted(overlapping_compounds2.items(), key=lambda item: item[1], reverse=True)[:10]

[(3, 128),
 (8, 128),
 (14, 128),
 (16, 128),
 (17, 128),
 (13, 125),
 (21, 125),
 (1, 4),
 (19, 4),
 (22, 4)]

In [22]:
def match_by_cid_and_closest_log_dose(dili, l1000_adata):
    l1000_obs = l1000_adata.obs.copy()
    l1000_obs['log_dose'] = np.log(l1000_obs['pert_dose_uM'])

    dili['log_dose'] = np.log(dili['CONCENTRATION_UM'])

    l1000_by_cid = {cid: grp for cid, grp in l1000_obs.groupby('pubchem_cid', observed=True)}

    matched_l1000_idx = []
    difference = []
    for _, row in dili.iterrows():
        l1000_grp = l1000_by_cid[row['pubchem_cid']]
        closest = (l1000_grp['log_dose'] - row['log_dose']).abs().idxmin()
        difference.append((l1000_grp['log_dose'] - row['log_dose']).abs().min())
        matched_l1000_idx.append(closest)

    dili['matched_l1000_idx'] = matched_l1000_idx
    dili['diff'] = difference
    return dili, l1000_obs

def return_embeddings(adata, dim=64,):
    adata_obs = adata.obs.copy()
    pca = PCA(n_components=dim)
    emb_logFC = pca.fit_transform(adata.layers['logFC'])
    
    pca = PCA(n_components=dim)
    emb_t = pca.fit_transform(adata.layers['t'])
    
    adata_obs['PCA.logFC'] = emb_logFC.tolist()
    adata_obs['PCA.t'] = emb_t.tolist()
    return adata_obs

In [23]:
to_check = sorted(overlapping_compounds1.items(), key=lambda item: item[1], reverse=True)[:10]
for item in to_check:
    l_id = item[0]
    compounds_overlapped = list(set(df_compounds['pubchem_cid']).intersection(l1000_phase1[l_id][l1000_phase1[l_id].obs['pert_time_h'] == 24].obs['pubchem_cid']))
    df_compounds_filtered = df_compounds[df_compounds['pubchem_cid'].isin(compounds_overlapped)].copy().dropna()

    df_compounds_filtered = df_compounds_filtered[(df_compounds_filtered['CONCENTRATION_UM'] < 15) & (df_compounds_filtered['CONCENTRATION_UM'] > 5)].copy()
    
    
    l1000_phase1_no_duplicates = l1000_phase1[l_id][~l1000_phase1[l_id].obs.index.duplicated()]
    l1000_phase1_filtered = l1000_phase1_no_duplicates[l1000_phase1_no_duplicates.obs['pubchem_cid'].isin(compounds_overlapped)]
    l1000_phase1_filtered_time = l1000_phase1_filtered[l1000_phase1_filtered.obs['pert_time_h'] == 24].copy()

    
    df_compounds_filtered_, _ = match_by_cid_and_closest_log_dose(df_compounds_filtered, l1000_phase1_filtered_time)
    l1000_phase1_filtered_time_obs = l1000_phase1_filtered_time.obs
    df_compounds_filtered_ = df_compounds_filtered_.merge(l1000_phase1_filtered_time_obs[['pert_dose_uM', 'pert_time_h']], how='left', left_on='matched_l1000_idx', right_index=True)
    
    print(l_id, len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique()))
    

47 81
14 81
25 76
54 73
5 62
37 59
62 56
0 56
43 56
7 46


In [24]:
to_check = sorted(overlapping_compounds2.items(), key=lambda item: item[1], reverse=True)[:10]
for item in to_check:
    l_id = item[0]
    compounds_overlapped = list(set(df_compounds['pubchem_cid']).intersection(l1000_phase2[l_id][l1000_phase2[l_id].obs['pert_time_h'] == 24].obs['pubchem_cid']))
    df_compounds_filtered = df_compounds[df_compounds['pubchem_cid'].isin(compounds_overlapped)].copy().dropna()

    df_compounds_filtered = df_compounds_filtered[(df_compounds_filtered['CONCENTRATION_UM'] < 15) & (df_compounds_filtered['CONCENTRATION_UM'] > 5)].copy()
    
    
    l1000_phase2_no_duplicates = l1000_phase2[l_id][~l1000_phase2[l_id].obs.index.duplicated()]
    l1000_phase2_filtered = l1000_phase2_no_duplicates[l1000_phase2_no_duplicates.obs['pubchem_cid'].isin(compounds_overlapped)]
    l1000_phase2_filtered_time = l1000_phase2_filtered[l1000_phase2_filtered.obs['pert_time_h'] == 24].copy()

    
    df_compounds_filtered_, _ = match_by_cid_and_closest_log_dose(df_compounds_filtered, l1000_phase2_filtered_time)
    l1000_phase2_filtered_time_obs = l1000_phase2_filtered_time.obs
    df_compounds_filtered_ = df_compounds_filtered_.merge(l1000_phase2_filtered_time_obs[['pert_dose_uM', 'pert_time_h']], how='left', left_on='matched_l1000_idx', right_index=True)
    
    print(l_id, len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique()))

3 59
8 59
14 59
16 59
17 59
13 59
21 59
1 1
19 1
22 1


# Processing

In [25]:
cols_l1000 = ['PCA.logFC', 
              'PCA.t', 
              'pert_dose_uM', 
              'pert_time_h']

In [26]:
rename_l1000 = {'pert_time_h': 'l1000_pert_time_h',
                'pert_dose_uM': 'l1000_pert_dose_uM'}

## Phase 1

In [27]:
l_id = 47

In [28]:
compounds_overlapped = list(set(df_compounds['pubchem_cid']).intersection(l1000_phase1[l_id][l1000_phase1[l_id].obs['pert_time_h'] == 24].obs['pubchem_cid']))

In [29]:
df_compounds_filtered = df_compounds[df_compounds['pubchem_cid'].isin(compounds_overlapped)].copy().dropna()
df_compounds_filtered = df_compounds_filtered[(df_compounds_filtered['CONCENTRATION_UM'] < 15) & (df_compounds_filtered['CONCENTRATION_UM'] > 5)].copy()
#df_compounds_filtered = df_compounds_filtered[df_compounds_filtered['CONCENTRATION_UM'] == 10].copy()

In [30]:
len(df_compounds_filtered['COMPOUND'].unique())

81

In [31]:
l1000_phase1_no_duplicates = l1000_phase1[l_id][~l1000_phase1[l_id].obs.index.duplicated()]
l1000_phase1_filtered = l1000_phase1_no_duplicates[l1000_phase1_no_duplicates.obs['pubchem_cid'].isin(compounds_overlapped)]
l1000_phase1_filtered_time = l1000_phase1_filtered[l1000_phase1_filtered.obs['pert_time_h'] == 24].copy()

df_compounds_filtered_, _ = match_by_cid_and_closest_log_dose(df_compounds_filtered, l1000_phase1_filtered_time)

## filter by prevailing dose

In [32]:
l1000_phase1_filtered_time_obs = l1000_phase1_filtered_time.obs

In [33]:
df_compounds_filtered_ = df_compounds_filtered_ = df_compounds_filtered_.merge(l1000_phase1_filtered_time_obs[['pert_dose_uM', 'pert_time_h']], how='left', left_on='matched_l1000_idx', right_index=True)

In [34]:
df_compounds_filtered_ = df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]

In [35]:
len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique())

81

In [36]:
df_compounds_filtered_ = df_compounds_filtered_.drop_duplicates(['COMPOUND', 'CONCENTRATION_UM'])

In [37]:
df_compounds_filtered_

,COMPOUND,CONCENTRATION_UM,TIMEPOINT_HOURS,SPLIT,pert_id,pubchem_cid,log_dose,matched_l1000_idx,diff,pert_dose_uM,pert_time_h
37,Indomethacin,10.000000,24,training,Indomethacin,3715,2.302585,3715_10uM_24h - 679_24h,0.000000,10.0,24.0
46,Phenylbutazone,8.000000,24,training,Phenylbutazone,4781,2.079442,4781_10uM_24h - 679_24h,0.223144,10.0,24.0
79,Chlorpromazine,10.000000,24,training,Chlorpromazine,2726,2.302585,2726_10uM_24h - 679_24h,0.000000,10.0,24.0
93,Chlorpheniramine,10.000000,24,training,Chlorpheniramine,2725,2.302585,2725_10uM_24h - 679_24h,0.000000,10.0,24.0
173,Diclofenac,11.000000,24,training,Diclofenac,3033,2.397895,3033_10uM_24h - 679_24h,0.095310,10.0,24.0
...,...,...,...,...,...,...,...,...,...,...,...
3967,Benztropine,10.000000,24,training,Benztropine,1201549,2.302585,1201549_10uM_24h - 679_24h,0.000000,10.0,24.0
4129,Entacapone,11.111111,24,training,Entacapone,5281081,2.407946,5281081_10uM_24h - 679_24h,0.105361,10.0,24.0
4131,Lapatinib,10.000000,24,training,Lapatinib,208908,2.302585,208908_10uM_24h - 679_24h,0.000000,10.0,24.0
4137,Levonorgestrel,5.555556,24,training,Levonorgestrel,13109,1.714798,13109_10uM_24h - 679_24h,0.587787,10.0,24.0


## 64

In [38]:
#V1
l1000_phase1_v1 = l1000_phase1_filtered_time[(l1000_phase1_filtered_time.obs.index.isin(df_compounds_filtered_['matched_l1000_idx']))]
l1000_phase1_obs_v1_64 = return_embeddings(l1000_phase1_v1)

print('matched doses:', l1000_phase1_obs_v1_64['pert_dose_uM'].unique())
dili_emb_v1_64_l1000_phase1 = df_compounds_filtered_.merge(l1000_phase1_obs_v1_64[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

matched doses: [10.]


In [39]:
#V2
l1000_phase1_v2 = l1000_phase1_no_duplicates.copy()
l1000_phase1_obs_v2_64 = return_embeddings(l1000_phase1_v2)
dili_emb_v2_64_l1000_phase1 = df_compounds_filtered_.merge(l1000_phase1_obs_v2_64[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

In [40]:
#V3
l1000_phase1_v3 = l1000_phase1_no_duplicates[(l1000_phase1_no_duplicates.obs['pert_dose_uM'].isin([10]))&(l1000_phase1_no_duplicates.obs['pert_time_h'].isin([24.]))].copy()
l1000_phase1_obs_v3_64 = return_embeddings(l1000_phase1_v3)
dili_emb_v3_64_l1000_phase1 = df_compounds_filtered_.merge(l1000_phase1_obs_v3_64[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

In [41]:
# V4
l1000_phase1_v4 = l1000_phase1_filtered.copy()
l1000_phase1_obs_v4_64 = return_embeddings(l1000_phase1_v4)
dili_emb_v4_64_l1000_phase1 = df_compounds_filtered_.merge(l1000_phase1_obs_v4_64[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

## 128

In [42]:
#V2
l1000_phase1_v2 = l1000_phase1_no_duplicates.copy()
l1000_phase1_obs_v2_128 = return_embeddings(l1000_phase1_v2, dim=128)
dili_emb_v2_128_l1000_phase1 = df_compounds_filtered_.merge(l1000_phase1_obs_v2_128[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

#V3
l1000_phase1_v3 = l1000_phase1_no_duplicates[(l1000_phase1_no_duplicates.obs['pert_dose_uM'].isin([10]))&(l1000_phase1_no_duplicates.obs['pert_time_h'].isin([24.]))].copy()
l1000_phase1_obs_v3_128 = return_embeddings(l1000_phase1_v3, dim=128)
dili_emb_v3_128_l1000_phase1 = df_compounds_filtered_.merge(l1000_phase1_obs_v3_128[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

# V4
l1000_phase1_v4 = l1000_phase1_filtered.copy()
l1000_phase1_obs_v4_128 = return_embeddings(l1000_phase1_v4, dim=128)
dili_emb_v4_128_l1000_phase1 = df_compounds_filtered_.merge(l1000_phase1_obs_v4_128[cols_l1000].rename(columns=rename_l1000), left_on='matched_l1000_idx', right_index=True, how='left')

In [43]:
dili_emb_v1_64_l1000_phase1['version'] = 'v1'
dili_emb_v2_64_l1000_phase1['version'] = 'v2'
dili_emb_v3_64_l1000_phase1['version'] = 'v3'
dili_emb_v4_64_l1000_phase1['version'] = 'v4'

dili_emb_v2_128_l1000_phase1['version'] = 'v2'
dili_emb_v3_128_l1000_phase1['version'] = 'v3'
dili_emb_v4_128_l1000_phase1['version'] = 'v4'

In [44]:
dili_emb_64_l1000_phase1 = pd.concat([dili_emb_v1_64_l1000_phase1,
           dili_emb_v2_64_l1000_phase1,
           dili_emb_v3_64_l1000_phase1,
           dili_emb_v4_64_l1000_phase1
          ])

In [45]:
dili_emb_128_l1000_phase1 = pd.concat([dili_emb_v2_128_l1000_phase1,
           dili_emb_v3_128_l1000_phase1,
           dili_emb_v4_128_l1000_phase1
          ])

In [46]:
dili_emb_64_l1000_phase1['dim'] = 64
dili_emb_128_l1000_phase1['dim'] = 128

dili_emb_64_l1000_phase1['l1000_phase'] = 1
dili_emb_128_l1000_phase1['l1000_phase'] = 1

In [47]:
dili_emb = pd.concat([dili_emb_64_l1000_phase1, 
          dili_emb_128_l1000_phase1])

In [48]:
dili_emb = dili_emb[~dili_emb['PCA.t'].isna()]

In [49]:
columns = ['COMPOUND', 
       'CONCENTRATION_UM', 'TIMEPOINT_HOURS',
        'SPLIT',
        'pubchem_cid', 'matched_l1000_idx',
        'l1000_pert_dose_uM', 'l1000_pert_time_h', 
        'PCA.logFC', 'PCA.t', 'version', 'dim',
       'l1000_phase']

In [50]:
dili_emb = dili_emb[columns].reset_index(drop=True).copy()

In [51]:
dili_emb

,COMPOUND,CONCENTRATION_UM,TIMEPOINT_HOURS,SPLIT,pubchem_cid,matched_l1000_idx,l1000_pert_dose_uM,l1000_pert_time_h,PCA.logFC,PCA.t,version,dim,l1000_phase
0,Indomethacin,10.000000,24,training,3715,3715_10uM_24h - 679_24h,10.0,24.0,"[0.4371413705174546, -3.3460436135755347, -1.0...","[2.670302238761189, -9.81847327107166, 22.6289...",v1,64,1
1,Phenylbutazone,8.000000,24,training,4781,4781_10uM_24h - 679_24h,10.0,24.0,"[0.8741563781981517, -0.08609445250827709, 1.6...","[1.796495398318519, 8.736892726509856, 2.24098...",v1,64,1
2,Chlorpromazine,10.000000,24,training,2726,2726_10uM_24h - 679_24h,10.0,24.0,"[0.1736672549884135, -0.9812549604150091, 2.01...","[-1.1927933831785178, 12.502880208198818, 7.31...",v1,64,1
3,Chlorpheniramine,10.000000,24,training,2725,2725_10uM_24h - 679_24h,10.0,24.0,"[2.072799641989336, -2.1450296814904593, -0.43...","[6.509206400676989, 4.659788485404951, -1.0414...",v1,64,1
4,Diclofenac,11.000000,24,training,3033,3033_10uM_24h - 679_24h,10.0,24.0,"[-0.08105772341696577, 1.3276836987006604, 0.1...","[-0.6932385669620214, -10.475022313646564, 4.1...",v1,64,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
562,Benztropine,10.000000,24,training,1201549,1201549_10uM_24h - 679_24h,10.0,24.0,"[-7.369634412311049, 2.590864394260027, -2.051...","[-8.1704046650406, -15.006339028511427, -4.254...",v4,128,1
563,Entacapone,11.111111,24,training,5281081,5281081_10uM_24h - 679_24h,10.0,24.0,"[4.57422441985542, -9.493693891167357, -0.6022...","[17.24855789481063, 10.836894810337137, 20.384...",v4,128,1
564,Lapatinib,10.000000,24,training,208908,208908_10uM_24h - 679_24h,10.0,24.0,"[-7.63407549938902, -3.4923173665157, -4.71331...","[-7.410080379847668, -23.843217189453263, 14.7...",v4,128,1
565,Levonorgestrel,5.555556,24,training,13109,13109_10uM_24h - 679_24h,10.0,24.0,"[-4.7972692619960595, -0.7773126147205643, -1....","[-10.504557715137004, -14.06845685022484, 2.21...",v4,128,1


In [52]:
dili_emb.to_pickle('dili_PCA_emb.pkl')